# SLT Grokking Experiment — Notebook Runner

**Supports Colab and Kaggle.** Set `PLATFORM = "colab"` or `"kaggle"` in Section 0.

| | Colab | Kaggle |
|---|---|---|
| GPU | T4 (free tier) | T4 / P100 (free tier) |
| Session limit | ~12 h (may disconnect) | 12 h (stable) |
| Persistence | Google Drive symlink | `/kaggle/working/` auto-saved as output |
| Cross-session restore | Drive (automatic) | Add previous output version as dataset input |

**Workflow per session:**
1. Run **Section 0** — set `PLATFORM`, clone GitHub, wire up persistence.
2. Run **Section 1** — install deps (once per session).
3. Run the section you need: Training / Calibration / LLC / Figures.

Repo: https://github.com/makataomu/slt-diplomka

## Section 0: Setup (run every session)

In [ ]:
import os, sys, shutil, subprocess

# ── CHOOSE YOUR PLATFORM ───────────────────────────────────────────────────────
PLATFORM = "kaggle"   # "colab"  or  "kaggle"
# ─────────────────────────────────────────────────────────────────────────────

REPO_URL = 'https://github.com/makataomu/slt-diplomka'

# ── Platform-specific paths ───────────────────────────────────────────────────
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR      = '/content/slt'
    PERSIST_DIR   = '/content/drive/MyDrive/slt_persist'
    for d in ['results/checkpoints', 'results/metrics', 'results/figures']:
        os.makedirs(f'{PERSIST_DIR}/{d}', exist_ok=True)

elif PLATFORM == "kaggle":
    REPO_DIR    = '/kaggle/working/slt'
    PERSIST_DIR = None   # /kaggle/working/ is auto-saved; no extra persistence dir needed

# ── Clone or pull from GitHub ─────────────────────────────────────────────────
if os.path.exists(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('Pulled latest from GitHub')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned from GitHub')

# ── Wire up results/ directory ────────────────────────────────────────────────
if PLATFORM == "colab":
    # Symlink results/ -> Drive so checkpoints/metrics persist across disconnects
    results_link = f'{REPO_DIR}/results'
    if os.path.islink(results_link):   os.unlink(results_link)
    elif os.path.isdir(results_link):  shutil.rmtree(results_link)
    os.symlink(f'{PERSIST_DIR}/results', results_link)
    print(f'Symlinked results/ -> {PERSIST_DIR}/results')

elif PLATFORM == "kaggle":
    # results/ sits inside /kaggle/working/slt/ — auto-saved at session end
    for sub in ['results/checkpoints', 'results/metrics', 'results/figures']:
        os.makedirs(f'{REPO_DIR}/{sub}', exist_ok=True)
    # Cross-session restore: if you added a previous output version as a dataset input,
    # it appears under /kaggle/input/<dataset-name>/slt/results/
    # This copies any missing checkpoint/metric files back into the working dir.
    for inp_name in sorted(os.listdir('/kaggle/input')):
        prev = f'/kaggle/input/{inp_name}/slt/results'
        if os.path.exists(prev):
            print(f'Restoring results from /kaggle/input/{inp_name}/slt/results ...')
            for sub in ['checkpoints', 'metrics']:
                src = f'{prev}/{sub}'
                dst = f'{REPO_DIR}/results/{sub}'
                if not os.path.exists(src):
                    continue
                for item in os.listdir(src):
                    s, d = f'{src}/{item}', f'{dst}/{item}'
                    if not os.path.exists(d):
                        (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
            print('  Done.')
            break
    print(f'Results: {REPO_DIR}/results/')

# ── Restore llc_calibration.yaml ─────────────────────────────────────────────
calib_local = f'{REPO_DIR}/configs/llc_calibration.yaml'
calib_src   = None

if PLATFORM == "colab":
    candidate = f'{PERSIST_DIR}/llc_calibration.yaml'
    if os.path.exists(candidate):
        calib_src = candidate

elif PLATFORM == "kaggle":
    # Check dataset inputs for a backed-up calibration yaml
    for inp_name in sorted(os.listdir('/kaggle/input')):
        candidate = f'/kaggle/input/{inp_name}/llc_calibration.yaml'
        if os.path.exists(candidate):
            calib_src = candidate
            break

if calib_src:
    shutil.copy(calib_src, calib_local)
    print(f'Restored llc_calibration.yaml from {calib_src}')
else:
    print('No saved calibration config found (defaults will be used until Section 3)')

# ── Set working directory and Python path ────────────────────────────────────
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

print(f'\nReady. Platform={PLATFORM}  |  Working dir: {os.getcwd()}')

## Section 1: Install dependencies (run once per session)

---
> **Kaggle users:** Before running Section 1, make sure **Internet** is enabled in the notebook settings (right panel → Internet → On). Without it, `pip install` and `git clone` will fail.

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
dv = importlib.metadata.version('devinterp')
print(f'devinterp {dv}  |  torch {torch.__version__}  |  device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print('LLC estimation: custom SGLD loop (devinterp.optim.SGLD) — bypasses v2 API')

## Section 2: Training

Run one `(ratio, seed)` pair. Set `RESUME = True` after a disconnect to continue from the last checkpoint.

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO  = 0.50   # one of [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEED   = 0      # 0, 1, or 2
RESUME = False  # set True to resume from last checkpoint
# ─────────────────────────────────────────────────────────────────────────────

resume_flag = '--resume' if RESUME else ''
!python src/train.py --ratio {RATIO} --seed {SEED} {resume_flag}

## Section 3: LLC Calibration

Run **once** after the ratio=0.50 seed=0 training run is done.
Inspect the chain traces, fill in the calibrated values, then run the save cell.
Good calibration: chains fluctuate in a stable band (not diverging, not flatlined).

In [ ]:
# Run calibration (uses final checkpoint of ratio=0.50 seed=0)
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate

In [ ]:
# Plot chain traces
import numpy as np
import matplotlib.pyplot as plt

traces = np.load('results/metrics/calibration_traces.npy')
print(f'Shape: {traces.shape}  (chains × draws)')

fig, ax = plt.subplots(figsize=(10, 4))
for i, chain in enumerate(traces):
    ax.plot(chain, lw=0.8, alpha=0.7, label=f'Chain {i}')
ax.set(xlabel='Draw', ylabel='Loss (SGLD)',
       title='SGLD calibration traces — chains should mix, not diverge or flatline')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('Chains diverge  → lower epsilon (e.g. 1e-5)')
print('Chains flatline → raise epsilon or num_draws')

In [ ]:
# Save calibrated hyperparams (edit values first, then run)
import yaml, shutil
from pathlib import Path

# ── FILL IN after inspecting traces above ─────────────────────────────────────
# nbeta starting point: default_nbeta(256) = 256/log(256) ≈ 46.2
# Adjust based on chain trace health (stable band = good)
CALIBRATED = dict(
    calibrated           = True,
    epsilon              = 1e-4,
    nbeta                = 46.2,   # default_nbeta(256); adjust if chains look bad
    gamma                = 10.0,
    num_chains           = 8,
    num_draws            = 500,
    num_burnin_steps     = 100,
    calibration_checkpoint = 'results/checkpoints/ratio_0.50/seed_0/epoch_06000.pt',
    calibration_date     = '2026-XX-XX',
    calibration_notes    = '',
)
# ─────────────────────────────────────────────────────────────────────────────

yaml_str = yaml.dump(CALIBRATED, default_flow_style=False)
Path('configs/llc_calibration.yaml').write_text(yaml_str)

# Back up so it survives the next session's git clone
if PLATFORM == "colab":
    shutil.copy('configs/llc_calibration.yaml', f'{PERSIST_DIR}/llc_calibration.yaml')
    print(f'Backed up to {PERSIST_DIR}/llc_calibration.yaml')
elif PLATFORM == "kaggle":
    # Saved to /kaggle/working/ — it appears as a session output automatically.
    # Next session: add this output version as a dataset input and Section 0 will restore it.
    shutil.copy('configs/llc_calibration.yaml', '/kaggle/working/llc_calibration.yaml')
    print('Saved to /kaggle/working/llc_calibration.yaml (will appear in session output)')

print('\n' + yaml_str)

## Section 4: LLC Estimation

Run after training + calibration. Processes all 100 checkpoints for one (ratio, seed).
Estimate: ~2–4 min per checkpoint × 100 = **3–7 hours per run**.

**Ask Tair before starting** — this is >2h compute.

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO = 0.50
SEED  = 0
# ─────────────────────────────────────────────────────────────────────────────

import yaml
cfg = yaml.safe_load(open('configs/llc_calibration.yaml'))
if not cfg.get('calibrated'):
    print('ERROR: run Section 3 (calibration) first.')
else:
    print(f'epsilon={cfg["epsilon"]}  nbeta={cfg["nbeta"]}  gamma={cfg["gamma"]}')
    !python src/llc_estimation.py --ratio {RATIO} --seed {SEED}

## Section 5: Generate Figures

In [ ]:
!python src/plotting.py

from pathlib import Path
for f in sorted(Path('results/figures').glob('*.pdf')):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

## Section 6: Full training sweep

Runs all 7 ratios × 3 seeds = **21 runs × ~30 min on T4 ≈ 10–11 hours**.

**On Kaggle:** fits comfortably within the 12-hour session limit. All checkpoints are auto-saved when you click *Save & Run All* or at session end.

**On Colab:** results persist on Drive, so disconnects are safe — just re-run Section 0 and this cell with `--resume` already on by default.

In [ ]:
RATIOS = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEEDS  = [0, 1, 2]

for ratio in RATIOS:
    for seed in SEEDS:
        print(f'\n=== ratio={ratio}  seed={seed} ===')
        !python src/train.py --ratio {ratio} --seed {seed} --epochs 6000 --checkpoint_every 60 --resume

print('\nAll training runs complete.')

## Section 7: Diagnostics — what's been computed so far

In [ ]:
from pathlib import Path
import pandas as pd

print('=== Checkpoints ===')
for ratio_dir in sorted(Path('results/checkpoints').glob('ratio_*')):
    for seed_dir in sorted(ratio_dir.glob('seed_*')):
        ckpts = sorted(seed_dir.glob('epoch_*.pt'))
        if ckpts:
            last = int(ckpts[-1].stem.split('_')[1])
            print(f'  {ratio_dir.name}/{seed_dir.name}: {len(ckpts)} checkpoints, last epoch={last}')

print('\n=== Training metrics ===')
for f in sorted(Path('results/metrics').glob('ratio_*.csv')):
    if '_llc' not in f.name:
        df = pd.read_csv(f)
        print(f'  {f.name}: {len(df)} rows, max_epoch={df["epoch"].max()}')

print('\n=== LLC metrics ===')
for f in sorted(Path('results/metrics').glob('*_llc.csv')):
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df)} checkpoints estimated')